# DBSCAN v2 — CWRU by_fault_type — Comparaison exp_079 vs exp_092

| Champ | Valeur |
|-------|--------|
| **Expérience ancienne** | exp_079 — eps=0.5 (bug: clé ignorée), seuil cross-tâche |
| **Expérience nouvelle** | exp_092 — eps knn_elbow auto, seuil par tâche |
| **Scénario** | by_fault_type : Ball → Inner Race → Outer Race (3 tâches) |
| **Sprint** | 13 — analyse paramètres non supervisés CWRU |

**Bug corrigé** : la config utilisait `eps: null` mais le code lisait `EPSILON` (uppercase) → jamais lu → eps=0.5 hardcodé. Correction : lecture de `eps` + estimation automatique via k-NN elbow.

> **Attention** : le résultat AA=89.6% de exp_092 est **artifactuel** (voir Section 5).

In [ ]:
import json, os, sys
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

_cwd = Path('.').resolve()
if _cwd.name == 'cwru_by_fault_type': os.chdir(_cwd.parent.parent.parent)
elif _cwd.name == 'cl_eval': os.chdir(_cwd.parent.parent)
elif _cwd.name == 'notebooks': os.chdir(_cwd.parent)
REPO_ROOT = Path('.').resolve()
if str(REPO_ROOT) not in sys.path: sys.path.insert(0, str(REPO_ROOT))

OLD_DIR = REPO_ROOT / 'experiments/exp_079_dbscan_cwru_by_fault_type/results'
NEW_DIR = REPO_ROOT / 'experiments/exp_092_dbscan_cwru_by_fault_type_v2/results'
FIGURES_DIR = REPO_ROOT / 'notebooks/figures/cl_evaluation/dbscan/cwru/by_fault_type_v2'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

TASK_NAMES = ['Ball', 'Inner Race', 'Outer Race']

old = json.loads((OLD_DIR / 'metrics_cl.json').read_text())
new = json.loads((NEW_DIR / 'metrics_cl.json').read_text())
print('Chargement OK')
print(f'eps_per_task (exp_092): {new["eps_per_task"]}')

In [ ]:
# Section 2 — Comparaison métriques

table = f"""
| Métrique | exp_079 (eps=0.5 bug, seuil cross) | exp_092 (eps knn_elbow, seuil par tâche) | Delta |
|----------|------------------------------------|-----------------------------------------|-------|
| **AA** | {old['acc_final']:.4f} | {new['acc_final']:.4f} | {new['acc_final']-old['acc_final']:+.4f} |
| **AF** | {old['avg_forgetting']:.4f} | {new['avg_forgetting']:.4f} | — |
| **BWT** | {old['backward_transfer']:+.4f} | {new['backward_transfer']:+.4f} | — |
| **RAM peak** | {old['ram_peak_bytes']:,} B ({old['ram_peak_bytes']/1024:.1f} Ko) | {new['ram_peak_bytes']:,} B ({new['ram_peak_bytes']/1024:.1f} Ko) | ⚠️ quasi limite |
| **n_params (core points)** | {old['n_params']:,} | {new['n_params']:,} | +{new['n_params']-old['n_params']:,} |
| **eps Task 0** | 0.5 (hardcodé par bug) | {new['eps_per_task'][0]:.4f} (knn_elbow) | — |
| **eps Task 1** | 0.5 | {new['eps_per_task'][1]:.4f} | — |
| **eps Task 2** | 0.5 | {new['eps_per_task'][2]:.4f} | — |
| **Seuil Task 0** | (cross-tâche) | {new['thresholds_per_task']['0']:.4f} | 0.0 = classification triviale |
| **Budget 64 Ko** | ✅ | {'⚠️' if new['ram_peak_bytes'] > 52000 else '✅'} ({new['ram_peak_bytes']/1024:.1f} Ko) | — |
"""
display(Markdown(table))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Gauche : eps par tâche
ax = axes[0]
ax.bar(TASK_NAMES, new['eps_per_task'], color=['#2196F3', '#FF9800', '#9C27B0'], edgecolor='black', linewidth=0.6)
ax.axhline(y=0.5, color='red', linestyle='--', linewidth=1.5, label='eps=0.5 (ancien bug)')
for i, v in enumerate(new['eps_per_task']):
    ax.text(i, v + 0.1, f'{v:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylabel('eps estimé (knn_elbow)')
ax.set_title('exp_092 — eps auto-estimé par tâche', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

# Droite : RAM par rapport au budget
ax = axes[1]
RAM_BUDGET = 65536
RAM_WARN = 52000
models = ['exp_079\n(eps=0.5)', 'exp_092\n(knn_elbow)', 'Budget\nSTM32N6']
rams = [old['ram_peak_bytes'], new['ram_peak_bytes'], RAM_BUDGET]
colors = ['#90CAF9', '#FF9800' if new['ram_peak_bytes'] > RAM_WARN else '#4CAF50', '#4CAF50']
bars = ax.bar(models, rams, color=colors, edgecolor='black', linewidth=0.6)
ax.axhline(y=RAM_BUDGET, color='red', linestyle='--', linewidth=1.5, label='Limite 64 Ko')
ax.axhline(y=RAM_WARN, color='orange', linestyle=':', linewidth=1, label='Avertissement 52 Ko')
for bar, val in zip(bars, rams):
    ax.text(bar.get_x() + bar.get_width()/2, val + 500, f'{val/1024:.1f} Ko', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('RAM peak (octets)')
ax.set_title('RAM peak vs budget STM32N6', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.3)

fig.tight_layout()
fig.savefig(FIGURES_DIR / 'eps_ram_comparison.png', dpi=120, bbox_inches='tight')
display(Image(str(FIGURES_DIR / 'eps_ram_comparison.png')))

## Section 4 — Diagnostic : pourquoi AA=89.6% est artifactuel

### Chaîne de causalité

```
1. eps_knn_elbow = 7.2 (très grand)
   → 535/536 samples deviennent des core points
   → presque tout le dataset est en cluster dense

2. Score anomalie = distance au core point le plus proche
   → 95% des samples d'entraînement ont score ≈ 0 (ils SONT core points)
   → Seuil au 95ème percentile = 0.0

3. Tout sample de test avec score > 0 → classé "anomalie" (label=1)
   → Quasi tous les samples test ont score > 0
   → Modèle prédit TOUT comme "anomalie"

4. CWRU test set = ~90% de samples défectueux (label=1)
   → Prédire "tout anomalie" donne 90% d'accuracy par déséquilibre de classes
```

**Ce n'est PAS un bon modèle** — c'est l'équivalent d'un classifieur constant prédisant toujours "défaut".

### Problème profond : knn_elbow surestime eps

Sur les features temporelles CWRU (9 features, gammes variables), la courbe des k-NN distances est convexe avec un coude très à droite, donnant un eps≈7 qui englobe quasiment tout le dataset dans un cluster unique.

### RAM = 56.3 Ko sur 64 Ko ⚠️

Avec n_core=4806 points (≈535 par tâche × 9 features × 4 bytes), DBSCAN frôle la limite STM32N6. Ce point est **bloquant** pour le déploiement embarqué.

## Section 5 — Conclusions DBSCAN sur CWRU

| | exp_079 | exp_092 | Verdict |
|-|---------|---------|--------|
| eps | 0.5 (bug) | 7.2 (auto) | Les deux sont inadaptés |
| AA | 12.5% | 89.6% (**artifactuel**) | Pas d'amélioration réelle |
| RAM | 16.8 Ko | 56.3 Ko | ⚠️ quasi limite 64 Ko |

**DBSCAN n'est pas adapté à CWRU** pour les raisons suivantes :
1. **eps impossible à calibrer sans labels** : trop petit → tout est bruit ; trop grand → tout est core point, seuil=0, classifieur trivial
2. **RAM trop élevée** : les core points accumulent ~56 Ko (88% du budget STM32N6)
3. **Limite fondamentale** : détection de densité sans connaissance des proportions normaux/défauts

**Recommandation** : exclure DBSCAN des baselines CWRU finales ; garder Mahalanobis (1.6 Ko RAM) et KMeans (5.4 Ko) comme baselines non supervisées.